<a href="https://colab.research.google.com/github/hanaa1r/bayan-nlp-hanaa1r/blob/main/notebooks/08_optimization_serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# اليوم الرابع — مختبر 7: تحسين الاستدلال والخدمة
## Day 4 — Lab 7: Inference Optimisation & Serving

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** 🟢 Core → 🔵 Explore → 🟣 Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/08_optimization_serving.ipynb)

**الهدف:** تثبيت workload وbudget، قياس PyTorch FP32، تصدير ONNX، تجربة dynamic INT8، قياس quality tax، ثم اختبار خدمة FastAPI داخل Colab.

**Goal:** build an honest baseline/candidate benchmark and a versioned, canary-tested API without paid hosting.


## قبل التشغيل: حد الادعاء | Evidence boundary

يعمل الدفتر افتراضيًا على `google/bert_uncased_L-2_H-128_A-2` بوصفه `SYSTEMS_SMOKE`. النموذج صغير وEnglish-only ورأس التصنيف المنشأ هنا غير مضبوط لمهمة بيان؛ وجود نص عربي يختبر Unicode وعقد الخدمة فقط. **لا تستخدم أرقامه كجودة بيان ولا كتسليم Gate D.**

لإكمال Gate D غيّر إعدادات الخلية التالية إلى `PROJECT_MODE=True`، واربط checkpoint بيان المضبوط وملف validation ثنائي اللغة. عندئذ يصبح `artefact_role=PROJECT_ARTIFACT` وتستخدم labels الفعلية لحساب Macro-F1.

| Mode | What it proves | Final Gate D? |
|---|---|---|
| `SYSTEMS_SMOKE` | export, ORT, quantisation attempt, parity, API contract | No |
| `PROJECT_ARTIFACT` | measured behaviour of your trained Bayan artefact | Yes |

جميع artefacts الكبيرة تُنشأ في `/content/bayan_day4_artifacts` ولا تُرفع إلى GitHub. نحفظ التقارير الصغيرة فقط.


In [1]:
# تثبيت النسخ المراجعة لليوم الرابع عند الحاجة فقط
import importlib.metadata
import os
import subprocess
import sys

os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("ORT_DISABLE_TELEMETRY", "1")

assert sys.version_info >= (3, 11), "Day 4 Core requires Python 3.11+ (use the current Colab runtime)."
REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "onnx": "1.22.0",
    "onnxruntime": "1.29.0",
    "fastapi": "0.141.1",
    "httpx2": "2.12.0",
    "psutil": "7.2.2",
}
needs_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *needs_install
    ])

installed = {name: importlib.metadata.version(name) for name in REQUIRED}
assert installed == REQUIRED, (installed, REQUIRED)
print("DAY4_SETUP=PASS", installed)


DAY4_SETUP=PASS {'transformers': '5.15.1', 'tokenizers': '0.22.2', 'onnx': '1.22.0', 'onnxruntime': '1.29.0', 'fastapi': '0.141.1', 'httpx2': '2.12.0', 'psutil': '7.2.2'}


In [2]:
import csv
import hashlib
import json
import os
import platform
import sys
import urllib.request
import uuid
from contextlib import asynccontextmanager
from pathlib import Path
from time import perf_counter_ns
from typing import Literal

import numpy as np
import onnx
import onnxruntime as ort
import psutil
import torch
from fastapi import FastAPI
from fastapi.testclient import TestClient
from onnxruntime.quantization import QuantType, quantize_dynamic
from onnxruntime.quantization.preprocess import quant_pre_process
from pydantic import BaseModel, Field, field_validator
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ort.disable_telemetry_events()

print("PYTHON", sys.version.split()[0])
print("TORCH", torch.__version__)
print("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
print("ORT_PROVIDERS", ort.get_available_providers())


PYTHON 3.13.15
TORCH 2.11.0+cpu
DEVICE cpu
ORT_PROVIDERS ['AzureExecutionProvider', 'CPUExecutionProvider']


## 0) إعداد مشروعك والميزانية — عدّل قبل القياس

في Core اترك `PROJECT_MODE=False`. لبوابة Gate D:

1. احفظ نموذج يوم 2 المضبوط وtokenizer في Drive باستخدام `save_pretrained`.
2. أنشئ CSV validation آمنًا بأعمدة `example_id,split,language,text,label`.
3. اكتب budget في commit قبل تشغيل candidates.
4. غيّر القيم أدناه ثم اختر **Runtime → Restart session and run all**.

لا تجعل `PROJECT_MODEL_SOURCE` رابط Drive عامًا، ولا تضع الأوزان في GitHub.


In [3]:
# ===== Student configuration =====
# Gate-D automatically uses the trained project artifact produced by
# scripts/train_project_artifact.py. It refuses to claim PROJECT_ARTIFACT when
# either the model or validation contract is absent.
from pathlib import Path

PROJECT_ROOT = (
    Path("/content/bayan-nlp-hanaa1r")
    if Path("/content/bayan-nlp-hanaa1r").is_dir()
    else Path.cwd()
)
if PROJECT_ROOT != Path.cwd():
    import os
    os.chdir(PROJECT_ROOT)
DEFAULT_PROJECT_MODEL = PROJECT_ROOT / "artifacts/project-v1/topic"
DEFAULT_PROJECT_VALIDATION = PROJECT_ROOT / "artifacts/project-v1/topic_validation.csv"

PROJECT_MODE = DEFAULT_PROJECT_MODEL.is_dir() and DEFAULT_PROJECT_VALIDATION.is_file()
PROJECT_MODEL_SOURCE = str(DEFAULT_PROJECT_MODEL) if PROJECT_MODE else ""
PROJECT_TOKENIZER_SOURCE = str(DEFAULT_PROJECT_MODEL) if PROJECT_MODE else ""
PROJECT_VALIDATION_CSV = str(DEFAULT_PROJECT_VALIDATION) if PROJECT_MODE else ""
PROJECT_PREPROCESSING_VERSION = "ar-en-v1"

# Written before candidate measurement. The official HTTP concurrency gate is
# checked separately at p99 <= 40 ms; this model-only budget is deliberately
# conservative for a shared Colab CPU.
PERFORMANCE_BUDGET = {
    "max_p95_ms": 250.0,
    "min_throughput_items_s": 10.0,
    "max_quality_tax": 0.05,
    "target_device": "colab-cpu",
}
BUDGET_PROVENANCE = (
    "STUDENT_DEFINED_BEFORE_MEASUREMENT"
    if PROJECT_MODE else "COURSE_EXAMPLE_FOR_SYSTEMS_SMOKE"
)

WARMUP = 5
REPETITIONS = 30
MAX_LENGTH = 96
BATCH_SIZE = 4

if PROJECT_MODE:
    assert Path(PROJECT_MODEL_SOURCE).is_dir(), "Train the project artifact first"
    assert Path(PROJECT_VALIDATION_CSV).is_file(), "Missing validation workload"
    ARTEFACT_ROLE = "PROJECT_ARTIFACT"
    RESULT_LABEL = "MEASURED"
else:
    ARTEFACT_ROLE = "SYSTEMS_SMOKE"
    RESULT_LABEL = "SYSTEMS_SMOKE"

assert WARMUP >= 1 and REPETITIONS >= 30
print("PROJECT_MODE", PROJECT_MODE)
print("ARTEFACT_ROLE", ARTEFACT_ROLE)
print("BUDGET_PROVENANCE", BUDGET_PROVENANCE)
print("TARGET", PERFORMANCE_BUDGET)


ARTEFACT_ROLE SYSTEMS_SMOKE
BUDGET_PROVENANCE COURSE_EXAMPLE_FOR_SYSTEMS_SMOKE
TARGET {'max_p95_ms': 1000.0, 'min_throughput_items_s': 0.1, 'max_quality_tax': 0.05, 'target_device': 'colab-cpu'}


## 1) جلب وحدات الدورة القابلة للاختبار

إذا شغلت notebook من GitHub مباشرة فلن يكون `src/bayan` موجودًا في `/content`. تحاول الخلية استخدام نسخة مشروعك أولًا، وإلا تنزّل وحدتي القياس والخدمة النصيتين من مستودع الدورة. لا يوجد token أو API مدفوع.


In [4]:
COURSE_RAW = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main"
local_src = Path("src")
if not (local_src / "bayan" / "benchmarking.py").exists():
    local_src = Path("_bayan_course_src")
    package_dir = local_src / "bayan"
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / "__init__.py").write_text("", encoding="utf-8")
    for module in ("benchmarking.py", "serving.py"):
        destination = package_dir / module
        urllib.request.urlretrieve(
            f"{COURSE_RAW}/src/bayan/{module}", destination
        )
sys.path.insert(0, str(local_src.resolve()))

from bayan.benchmarking import (
    artifact_size_mb,
    assess_budget,
    benchmark_callable,
    quality_tax,
    speedup,
)
from bayan.serving import (
    ServingManifest,
    build_prediction_response,
    run_canaries,
    sha256_file,
    validate_manifest,
    validate_request_text,
)

print("BAYAN_HELPERS=PASS", local_src)


BAYAN_HELPERS=PASS _bayan_course_src


## 2) Workload ثابت وثنائي اللغة

في Systems Smoke نستخدم أمثلة صغيرة ثابتة ولا ندعي لها task labels. في Project Mode يجب أن تكون كل الصفوف `validation` أثناء اختيار candidate؛ لا تفتح frozen test حتى تثبت القرار.


In [5]:
SYSTEMS_ROWS = [
    {"example_id": "S-01", "split": "validation", "language": "ar", "text": "الخدمة الإلكترونية واضحة وسريعة", "label": ""},
    {"example_id": "S-02", "split": "validation", "language": "en", "text": "The online service is clear and fast", "label": ""},
    {"example_id": "S-03", "split": "validation", "language": "ar", "text": "تعذر تسجيل الدخول إلى البوابة", "label": ""},
    {"example_id": "S-04", "split": "validation", "language": "en", "text": "I cannot sign in to the portal", "label": ""},
    {"example_id": "S-05", "split": "validation", "language": "ar", "text": "أحتاج معرفة حالة طلب التصريح", "label": ""},
    {"example_id": "S-06", "split": "validation", "language": "en", "text": "I need the status of my permit request", "label": ""},
    {"example_id": "S-07", "split": "validation", "language": "ar", "text": "تأخر موعد العيادة هذا الصباح", "label": ""},
    {"example_id": "S-08", "split": "validation", "language": "en", "text": "My clinic appointment was delayed this morning", "label": ""},
]

if PROJECT_MODE:
    validation_path = Path(PROJECT_VALIDATION_CSV)
    assert validation_path.is_file(), validation_path
    with validation_path.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    required_columns = {"example_id", "split", "language", "text", "label"}
    assert rows and required_columns.issubset(rows[0]), required_columns
    assert all(row["split"] == "validation" for row in rows), (
        "Candidate selection must use validation only."
    )
    assert {"ar", "en"}.issubset({row["language"] for row in rows})
    assert all(row["label"].strip() for row in rows)
else:
    rows = SYSTEMS_ROWS

WORKLOAD_TEXTS = [row["text"] for row in rows]
workload_payload = json.dumps(rows, ensure_ascii=False, sort_keys=True).encode("utf-8")
WORKLOAD_SHA256 = hashlib.sha256(workload_payload).hexdigest()
assert len({row["example_id"] for row in rows}) == len(rows)
print("WORKLOAD", {"rows": len(rows), "languages": sorted({r["language"] for r in rows}), "sha256": WORKLOAD_SHA256})


WORKLOAD {'rows': 8, 'languages': ['ar', 'en'], 'sha256': '1d1d1c3bef8a582931f6a1c1803671e8fe194fc36ae59cdc4f4feca9fc4d6785'}


## 3) تحميل reference artefact

مسار Core يحمل BERT صغيرًا مرخصًا MIT لتقليل وقت التصدير. رأس التصنيف غير مضبوط؛ المخرجات لا تحمل معنى المهمة. Project Mode يحمل artefact المتدرب وlabel map المحفوظة معه.


In [6]:
if PROJECT_MODE:
    MODEL_SOURCE = PROJECT_MODEL_SOURCE
    TOKENIZER_SOURCE = PROJECT_TOKENIZER_SOURCE
else:
    MODEL_SOURCE = "google/bert_uncased_L-2_H-128_A-2"
    TOKENIZER_SOURCE = MODEL_SOURCE

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE, use_fast=True)
model_kwargs = {"attn_implementation": "eager"}
if not PROJECT_MODE:
    model_kwargs.update(num_labels=3, ignore_mismatched_sizes=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_SOURCE, **model_kwargs
)
model.eval()
device = torch.device("cpu")  # Same target for PyTorch and ORT Core comparison.
model.to(device)

LABEL_MAP = {int(key): str(value) for key, value in model.config.id2label.items()}
assert LABEL_MAP
state_digest = hashlib.sha256()
for parameter_name, tensor in sorted(model.state_dict().items()):
    state_digest.update(parameter_name.encode("utf-8"))
    state_digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
PYTORCH_STATE_SHA256 = state_digest.hexdigest()
PYTORCH_PARAMETER_SIZE_MIB = sum(
    tensor.numel() * tensor.element_size() for tensor in model.state_dict().values()
) / (1024 ** 2)
print("MODEL_SOURCE", MODEL_SOURCE)
print("TOKENIZER_SOURCE", TOKENIZER_SOURCE)
print("LABEL_MAP", LABEL_MAP)
print("PYTORCH_REFERENCE", {"sha256": PYTORCH_STATE_SHA256, "parameter_size_mib": round(PYTORCH_PARAMETER_SIZE_MIB, 3)})


config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.7MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MODEL_SOURCE google/bert_uncased_L-2_H-128_A-2
TOKENIZER_SOURCE google/bert_uncased_L-2_H-128_A-2
LABEL_MAP {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}
PYTORCH_REFERENCE {'sha256': 'b413ec986cbb7c7c4c3c1a6da3e088299f0848bc6d3eb45af917b821d88774df', 'parameter_size_mib': 16.732}


## 4) Length audit وdynamic padding

نقرأ الطول قبل اختيار `max_length`. في مشروعك بدّل القيمة بعد فحص truncation والشرائح، لا لأن 512 قيمة مشهورة.


In [7]:
token_lists = tokenizer(WORKLOAD_TEXTS, add_special_tokens=True, truncation=False)["input_ids"]
token_lengths = np.asarray([len(tokens) for tokens in token_lists])
length_report = {
    "p50_tokens": float(np.percentile(token_lengths, 50)),
    "p95_tokens": float(np.percentile(token_lengths, 95)),
    "max_tokens": int(token_lengths.max()),
    "configured_max_length": MAX_LENGTH,
    "would_truncate": int((token_lengths > MAX_LENGTH).sum()),
}
assert length_report["would_truncate"] == 0 or PROJECT_MODE, (
    "Systems fixture should not truncate; inspect your project truncation deliberately."
)

encoded_batches = []
for start in range(0, len(WORKLOAD_TEXTS), BATCH_SIZE):
    batch = tokenizer(
        WORKLOAD_TEXTS[start:start + BATCH_SIZE],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded_batches.append({
        key: value.to(device)
        for key, value in batch.items()
        if key in {"input_ids", "attention_mask"}
    })
encoded = encoded_batches[0]  # Example inputs for export; dynamic axes handle later batches.
assert sum(batch["input_ids"].shape[0] for batch in encoded_batches) == len(WORKLOAD_TEXTS)
print("LENGTH_REPORT", length_report)
print("DYNAMIC_BATCH_SHAPES", [tuple(batch["input_ids"].shape) for batch in encoded_batches])


LENGTH_REPORT {'p50_tokens': 18.0, 'p95_tokens': 28.95, 'max_tokens': 30, 'configured_max_length': 96, 'would_truncate': 0}
DYNAMIC_BATCH_SHAPES [(4, 30), (4, 26)]


## 5) PyTorch FP32 baseline

`model.eval()` و`torch.inference_mode()` مختلفان ونستخدمهما معًا. نقيس model-only على batch ثابت؛ ثم نقيس end-to-end منفصلًا حتى لا نقارن حدودًا مختلفة.


In [8]:
try:
    process = psutil.Process(os.getpid())
    process.memory_info()
    memory_reader = lambda: process.memory_info().rss
    MEMORY_METHOD = "process RSS start and observed peak; approximate"
except (psutil.Error, OSError) as exc:
    process = None
    memory_reader = None
    MEMORY_METHOD = f"RSS unavailable in this runtime: {type(exc).__name__}"
if PROJECT_MODE:
    assert memory_reader is not None, "Gate D requires a documented memory measurement."

def pytorch_logits_from_encoded():
    outputs = []
    with torch.inference_mode():
        for batch in encoded_batches:
            outputs.append(model(**batch).logits.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)

def pytorch_end_to_end():
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(WORKLOAD_TEXTS), BATCH_SIZE):
            fresh = tokenizer(
                WORKLOAD_TEXTS[start:start + BATCH_SIZE],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            fresh = {
                key: value.to(device)
                for key, value in fresh.items()
                if key in {"input_ids", "attention_mask"}
            }
            outputs.append(model(**fresh).logits.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)

baseline_logits = pytorch_logits_from_encoded()
baseline_predictions = baseline_logits.argmax(axis=1)
baseline_model_report = benchmark_callable(
    pytorch_logits_from_encoded,
    warmup=WARMUP,
    repetitions=REPETITIONS,
    items_per_call=len(baseline_predictions),
    memory_reader=memory_reader,
)
baseline_e2e_report = benchmark_callable(
    pytorch_end_to_end,
    warmup=WARMUP,
    repetitions=REPETITIONS,
    items_per_call=len(baseline_predictions),
    memory_reader=memory_reader,
)
print("PYTORCH_MODEL_ONLY", {"p95_ms": round(baseline_model_report["p95_ms"], 3), "warmup": WARMUP, "repetitions": REPETITIONS, "items_per_call": len(WORKLOAD_TEXTS)})
print("PYTORCH_END_TO_END", {"measured": True, "boundary": "tokenisation + model", "saved_in": "reports/benchmark_results.json"})


PYTORCH_MODEL_ONLY {'p95_ms': 37.554, 'warmup': 5, 'repetitions': 30, 'items_per_call': 8}
PYTORCH_END_TO_END {'measured': True, 'boundary': 'tokenisation + model', 'saved_in': 'reports/benchmark_results.json'}


## 6) تصدير ONNX FP32 والتحقق

نستخدم encoded inputs نفسها، ثم نشغل `onnx.checker`. لا تعني عبارة export completed أن parity نجحت.

> **ملاحظة توافق:** يثبت Core مسار TorchScript exporter صراحةً (`dynamo=False`) لأن dynamic INT8 في ONNX Runtime لا يدعم كل graphs الناتجة من المصدّر الجديد بعد. قد يظهر تحذير deprecation أو tracing؛ لا تعتبره نجاحًا أو فشلًا. الحكم هو ONNX checker ثم numerical/prediction parity. المصدّر الجديد مسار Explore بعد نجاح Core.


In [9]:
ARTIFACT_DIR = Path("/content/bayan_day4_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FP32_ONNX = ARTIFACT_DIR / "bayan_model_fp32.onnx"
FP32_ONNX_PREPROCESSED = ARTIFACT_DIR / "bayan_model_fp32_preprocessed.onnx"
INT8_ONNX = ARTIFACT_DIR / "bayan_model_dynamic_int8.onnx"

class LogitsWrapper(torch.nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, input_ids, attention_mask):
        return self.inner(input_ids=input_ids, attention_mask=attention_mask).logits

wrapper = LogitsWrapper(model).eval()
torch.onnx.export(
    wrapper,
    (encoded["input_ids"], encoded["attention_mask"]),
    str(FP32_ONNX),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
onnx_model = onnx.load(str(FP32_ONNX))
onnx.checker.check_model(onnx_model)
assert FP32_ONNX.is_file() and FP32_ONNX.stat().st_size > 0
quant_pre_process(
    str(FP32_ONNX), str(FP32_ONNX_PREPROCESSED), skip_symbolic_shape=True
)
assert FP32_ONNX_PREPROCESSED.is_file() and FP32_ONNX_PREPROCESSED.stat().st_size > 0
print("ONNX_CHECKER=PASS", FP32_ONNX.name, round(artifact_size_mb(FP32_ONNX), 3), "MiB")
print("QUANT_PREPROCESS=PASS", FP32_ONNX_PREPROCESSED.name)


/tmp/ipykernel_2395/986061057.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:603: TracerWarning: torch.tensor results are registered as constants in t

ONNX_CHECKER=PASS bayan_model_fp32.onnx 16.788 MiB
QUANT_PREPROCESS=PASS bayan_model_fp32_preprocessed.onnx


In [10]:
def ort_inputs(batch):
    return {
        "input_ids": batch["input_ids"].detach().cpu().numpy().astype(np.int64),
        "attention_mask": batch["attention_mask"].detach().cpu().numpy().astype(np.int64),
    }

fp32_session = ort.InferenceSession(
    str(FP32_ONNX), providers=["CPUExecutionProvider"]
)
expected_inputs = {item.name for item in fp32_session.get_inputs()}
assert expected_inputs == {"input_ids", "attention_mask"}, expected_inputs

def fp32_ort_logits():
    return np.concatenate([
        fp32_session.run(["logits"], ort_inputs(batch))[0]
        for batch in encoded_batches
    ], axis=0)

fp32_logits = fp32_ort_logits()
fp32_predictions = fp32_logits.argmax(axis=1)
fp32_parity = {
    "max_abs_logits_diff": float(np.max(np.abs(baseline_logits - fp32_logits))),
    "mean_abs_logits_diff": float(np.mean(np.abs(baseline_logits - fp32_logits))),
    "prediction_agreement": float(np.mean(baseline_predictions == fp32_predictions)),
}
assert fp32_parity["max_abs_logits_diff"] < 1e-3, fp32_parity
assert fp32_parity["prediction_agreement"] == 1.0, fp32_parity

fp32_ort_report = benchmark_callable(
    fp32_ort_logits,
    warmup=WARMUP,
    repetitions=REPETITIONS,
    items_per_call=len(fp32_predictions),
    memory_reader=memory_reader,
)
print("ONNX_FP32_PARITY=PASS", {"max_abs_logits_diff": fp32_parity["max_abs_logits_diff"], "prediction_agreement": fp32_parity["prediction_agreement"]})
print("ONNX_FP32_MODEL_ONLY", {"p95_ms": round(fp32_ort_report["p95_ms"], 3), "warmup": WARMUP, "repetitions": REPETITIONS})


ONNX_FP32_PARITY=PASS {'max_abs_logits_diff': 2.0489096641540527e-07, 'prediction_agreement': 1.0}
ONNX_FP32_MODEL_ONLY {'p95_ms': 24.653, 'warmup': 5, 'repetitions': 30}


## 7) Dynamic INT8 candidate

نقوم بمحاولة فعلية ثم نوثق النتيجة. إذا كان graph/operator غير مدعوم لا نطبع PASS مزيفًا: نحتفظ بـONNX FP32 ونضع سبب الفشل في التقرير.


In [11]:
int8_status = {"attempted": True, "available": False, "error": None}
int8_session = None
int8_report = None
int8_parity = None
try:
    quantize_dynamic(
        model_input=str(FP32_ONNX_PREPROCESSED),
        model_output=str(INT8_ONNX),
        weight_type=QuantType.QInt8,
    )
    int8_session = ort.InferenceSession(
        str(INT8_ONNX), providers=["CPUExecutionProvider"]
    )

    def int8_ort_logits():
        return np.concatenate([
            int8_session.run(["logits"], ort_inputs(batch))[0]
            for batch in encoded_batches
        ], axis=0)

    int8_logits = int8_ort_logits()
    int8_predictions = int8_logits.argmax(axis=1)
    int8_parity = {
        "max_abs_logits_diff": float(np.max(np.abs(baseline_logits - int8_logits))),
        "mean_abs_logits_diff": float(np.mean(np.abs(baseline_logits - int8_logits))),
        "prediction_agreement": float(np.mean(baseline_predictions == int8_predictions)),
    }
    int8_report = benchmark_callable(
        int8_ort_logits,
        warmup=WARMUP,
        repetitions=REPETITIONS,
        items_per_call=len(int8_predictions),
        memory_reader=memory_reader,
    )
    int8_status["available"] = True
    print("INT8_ATTEMPT=PASS", round(artifact_size_mb(INT8_ONNX), 3), "MiB")
    print("INT8_PARITY", {"prediction_agreement": int8_parity["prediction_agreement"]})
    print("INT8_MODEL_ONLY", {"p95_ms": round(int8_report["p95_ms"], 3), "warmup": WARMUP, "repetitions": REPETITIONS})
except Exception as exc:
    int8_status["error"] = f"{type(exc).__name__}: {exc}"
    print("INT8_ATTEMPT=DOCUMENTED_UNSUPPORTED", int8_status["error"])

assert int8_status["attempted"] is True
assert int8_status["available"] or int8_status["error"]


INT8_ATTEMPT=PASS 4.287 MiB
INT8_PARITY {'prediction_agreement': 0.625}
INT8_MODEL_ONLY {'p95_ms': 10.031, 'warmup': 5, 'repetitions': 30}


## 8) Quality tax وقرار budget

في Systems Smoke نستخدم agreement مع FP32، لا task quality. في Project Mode نحسب Macro-F1 من labels الفعلية على workload validation كاملة، وتمر الدالة على batches مع dynamic padding.

قيمة `BATCH_SIZE` جزء من عقد القياس؛ لا تغيرها بين baseline وcandidate.


In [12]:
def macro_f1(y_true, y_pred):
    labels = sorted(set(y_true) | set(y_pred))
    scores = []
    for label in labels:
        tp = sum(t == label and p == label for t, p in zip(y_true, y_pred))
        fp = sum(t != label and p == label for t, p in zip(y_true, y_pred))
        fn = sum(t == label and p != label for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(scores)) if scores else 0.0

if PROJECT_MODE:
    y_true = [row["label"] for row in rows]
    baseline_labels = [LABEL_MAP[int(index)] for index in baseline_predictions]
    fp32_labels = [LABEL_MAP[int(index)] for index in fp32_predictions]
    baseline_quality = macro_f1(y_true, baseline_labels)
    fp32_quality = macro_f1(y_true, fp32_labels)
    quality_metric = "macro_f1_validation_full_workload"
else:
    baseline_quality = 1.0
    fp32_quality = fp32_parity["prediction_agreement"]
    quality_metric = "prediction_agreement_to_fp32_not_task_quality"

fp32_quality_tax = quality_tax(baseline_quality, fp32_quality)
fp32_budget = assess_budget(
    fp32_ort_report,
    quality_tax_value=fp32_quality_tax,
    max_p95_ms=PERFORMANCE_BUDGET["max_p95_ms"],
    max_quality_tax=PERFORMANCE_BUDGET["max_quality_tax"],
    min_throughput_items_s=PERFORMANCE_BUDGET["min_throughput_items_s"],
)

if int8_status["available"]:
    if PROJECT_MODE:
        int8_labels = [LABEL_MAP[int(index)] for index in int8_predictions]
        int8_quality = macro_f1(y_true, int8_labels)
    else:
        int8_quality = int8_parity["prediction_agreement"]
    int8_quality_tax = quality_tax(baseline_quality, int8_quality)
    int8_budget = assess_budget(
        int8_report,
        quality_tax_value=int8_quality_tax,
        max_p95_ms=PERFORMANCE_BUDGET["max_p95_ms"],
        max_quality_tax=PERFORMANCE_BUDGET["max_quality_tax"],
        min_throughput_items_s=PERFORMANCE_BUDGET["min_throughput_items_s"],
    )
else:
    int8_quality = None
    int8_quality_tax = None
    int8_budget = None

if int8_status["available"] and int8_budget["budget_met"]:
    selected_name = "onnx-dynamic-int8"
    selected_session = int8_session
    selected_sha256 = sha256_file(str(INT8_ONNX))
    adoption_decision = "ADOPT_INT8"
elif fp32_budget["budget_met"]:
    selected_name = "onnx-fp32"
    selected_session = fp32_session
    selected_sha256 = sha256_file(str(FP32_ONNX))
    adoption_decision = "ADOPT_ONNX_FP32"
else:
    selected_name = "pytorch-fp32"
    selected_session = None
    selected_sha256 = PYTORCH_STATE_SHA256
    adoption_decision = "KEEP_PYTORCH_FP32"

ship_decision = (
    "PROJECT_BUDGET_DECISION" if PROJECT_MODE else "SYSTEMS_SMOKE_NOT_A_SHIP_DECISION"
)
print("QUALITY_METRIC", quality_metric)
print("FP32_ORT_BUDGET", fp32_budget)
print("INT8_BUDGET", int8_budget)
print("SELECTED_FOR_SERVICE", selected_name, adoption_decision, ship_decision)


QUALITY_METRIC prediction_agreement_to_fp32_not_task_quality
FP32_ORT_BUDGET {'latency_ok': True, 'quality_ok': True, 'throughput_ok': True, 'budget_met': True}
INT8_BUDGET {'latency_ok': True, 'quality_ok': False, 'throughput_ok': True, 'budget_met': False}
SELECTED_FOR_SERVICE onnx-fp32 ADOPT_ONNX_FP32 SYSTEMS_SMOKE_NOT_A_SHIP_DECISION


## 9) Manifest وخدمة FastAPI

الاستجابة توضح model/runtime/preprocessing version. في مشروعك ثبّت expected canary labels في ملف versioned بعد قبول artefact؛ توليدها آليًا هنا مقبول فقط لـSystems Smoke.


In [13]:
manifest = ServingManifest(
    model_id=str(MODEL_SOURCE),
    model_version="project-v1" if PROJECT_MODE else "systems-smoke-v1",
    preprocessing_version=PROJECT_PREPROCESSING_VERSION,
    runtime=selected_name,
    label_map=LABEL_MAP,
    artifact_sha256=selected_sha256,
)
validate_manifest(
    manifest,
    expected_preprocessing_version=PROJECT_PREPROCESSING_VERSION,
)

def softmax(array):
    shifted = array - np.max(array, axis=-1, keepdims=True)
    values = np.exp(shifted)
    return values / values.sum(axis=-1, keepdims=True)

def service_predict(text, language="auto"):
    clean = validate_request_text(text, max_chars=1000)
    item = tokenizer(
        clean,
        padding=False,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    inputs = {
        "input_ids": item["input_ids"].astype(np.int64),
        "attention_mask": item["attention_mask"].astype(np.int64),
    }
    start = perf_counter_ns()
    if selected_session is not None:
        logits = selected_session.run(["logits"], inputs)[0]
    else:
        torch_inputs = {key: torch.from_numpy(value).to(device) for key, value in inputs.items()}
        with torch.inference_mode():
            logits = model(**torch_inputs).logits.detach().cpu().numpy()
    latency_ms = (perf_counter_ns() - start) / 1_000_000
    probabilities = softmax(logits)[0]
    index = int(probabilities.argmax())
    return build_prediction_response(
        request_id=str(uuid.uuid4()),
        text=clean,
        language=language,
        label=LABEL_MAP[index],
        confidence=float(probabilities[index]),
        latency_ms=latency_ms,
        manifest=manifest,
    )

seed_canaries = [
    {"name": "arabic-contract", "text": "الخدمة واضحة", "language": "ar"},
    {"name": "english-contract", "text": "The service is clear", "language": "en"},
]
for case in seed_canaries:
    case["expected_label"] = service_predict(case["text"], case["language"])["prediction"]["label"]
print("CANARY_EXPECTATIONS", seed_canaries)


CANARY_EXPECTATIONS [{'name': 'arabic-contract', 'text': 'الخدمة واضحة', 'language': 'ar', 'expected_label': 'LABEL_2'}, {'name': 'english-contract', 'text': 'The service is clear', 'language': 'en', 'expected_label': 'LABEL_2'}]


In [14]:
class ClassifyRequest(BaseModel):
    text: str = Field(min_length=1, max_length=1000)
    language: Literal["ar", "en", "auto"] = "auto"

    @field_validator("text")
    @classmethod
    def reject_blank_text(cls, value):
        return validate_request_text(value, max_chars=1000)


@asynccontextmanager
async def lifespan(app):
    validate_manifest(
        manifest,
        expected_preprocessing_version=PROJECT_PREPROCESSING_VERSION,
    )
    app.state.canary_report = run_canaries(service_predict, seed_canaries)
    app.state.ready = True
    yield
    app.state.ready = False


app = FastAPI(title="Bayan Classification API", version="1.0.0", lifespan=lifespan)


@app.get("/health")
def health():
    return {
        "status": "ready" if app.state.ready else "not_ready",
        "model_id": manifest.model_id,
        "model_version": manifest.model_version,
        "runtime": manifest.runtime,
        "preprocessing_version": manifest.preprocessing_version,
        "canaries": app.state.canary_report,
    }


@app.post("/v1/classify")
def classify(request: ClassifyRequest):
    return service_predict(request.text, request.language)

print("FASTAPI_APP=BUILT")


FASTAPI_APP=BUILT


## 10) TestClient: عربي + إنجليزي + رفض input

لا نفتح tunnel ولا ننشر Colab. TestClient يثبت عقد HTTP داخل runtime. status 422 هو السلوك الصحيح لطلب لا يطابق schema.


In [15]:
with TestClient(app) as client:
    health_response = client.get("/health")
    arabic_response = client.post(
        "/v1/classify", json={"text": "الخدمة واضحة", "language": "ar"}
    )
    english_response = client.post(
        "/v1/classify", json={"text": "The service is clear", "language": "en"}
    )
    empty_response = client.post(
        "/v1/classify", json={"text": "   ", "language": "auto"}
    )
    language_response = client.post(
        "/v1/classify", json={"text": "valid text", "language": "fr"}
    )

assert health_response.status_code == 200
assert health_response.json()["status"] == "ready"
assert len(health_response.json()["canaries"]) == 2
assert arabic_response.status_code == 200 and arabic_response.json()["language"] == "ar"
assert english_response.status_code == 200 and english_response.json()["language"] == "en"
assert empty_response.status_code == 422
assert language_response.status_code == 422
assert arabic_response.json()["model"]["preprocessing_version"] == PROJECT_PREPROCESSING_VERSION
service_test_summary = {
    "health": health_response.status_code,
    "arabic": arabic_response.status_code,
    "english": english_response.status_code,
    "empty_rejected": empty_response.status_code,
    "unsupported_language_rejected": language_response.status_code,
    "canaries": health_response.json()["canaries"],
}
print("FASTAPI_TESTCLIENT=PASS", service_test_summary)


FASTAPI_TESTCLIENT=PASS {'health': 200, 'arabic': 200, 'english': 200, 'empty_rejected': 422, 'unsupported_language_rejected': 422, 'canaries': [{'name': 'arabic-contract', 'status': 'PASS', 'label': 'LABEL_2'}, {'name': 'english-contract', 'status': 'PASS', 'label': 'LABEL_2'}]}


## 11) حفظ تقارير صغيرة قابلة للمراجعة

التقرير يسجل أن RSS observed تقريبي، وأن Systems Smoke ليس benchmark مشروع. لا تحفظ ملفات ONNX في GitHub؛ احتفظ بـhash وخطوات إعادة الإنتاج.


In [16]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "onnx": onnx.__version__,
    "onnxruntime": ort.__version__,
    "device": str(device),
    "ort_provider": "CPUExecutionProvider",
}
benchmark_results = {
    "result_label": RESULT_LABEL,
    "artefact_role": ARTEFACT_ROLE,
    "warning": (
        "SYSTEMS_SMOKE proves mechanics only; replace with PROJECT_ARTIFACT for Gate D."
        if not PROJECT_MODE else None
    ),
    "environment": environment,
    "workload": {
        "rows": len(rows),
        "rows_measured_per_repetition": len(rows),
        "batch_size": BATCH_SIZE,
        "languages": sorted({row["language"] for row in rows}),
        "sha256": WORKLOAD_SHA256,
        "length": length_report,
    },
    "budget": PERFORMANCE_BUDGET,
    "budget_provenance": BUDGET_PROVENANCE,
    "measurement": {
        "warmup": WARMUP,
        "repetitions": REPETITIONS,
        "boundary": "model_only_primary_and_pytorch_end_to_end_secondary",
        "memory_method": MEMORY_METHOD,
    },
    "pytorch_fp32": {
        "model_only": baseline_model_report,
        "end_to_end": baseline_e2e_report,
        "parameter_size_mib": PYTORCH_PARAMETER_SIZE_MIB,
        "state_sha256": PYTORCH_STATE_SHA256,
        "quality_metric": quality_metric,
        "quality": baseline_quality,
    },
    "onnx_fp32": {
        "model_only": fp32_ort_report,
        "parity": fp32_parity,
        "size_mib": artifact_size_mb(FP32_ONNX),
        "sha256": sha256_file(str(FP32_ONNX)),
        "quality": fp32_quality,
        "quality_tax": fp32_quality_tax,
        "budget": fp32_budget,
        "p95_speedup_vs_pytorch": speedup(
            baseline_model_report["p95_ms"], fp32_ort_report["p95_ms"]
        ),
    },
    "onnx_dynamic_int8": {
        "status": int8_status,
        "model_only": int8_report,
        "parity": int8_parity,
        "size_mib": artifact_size_mb(INT8_ONNX) if int8_status["available"] else None,
        "sha256": sha256_file(str(INT8_ONNX)) if int8_status["available"] else None,
        "quality": int8_quality,
        "quality_tax": int8_quality_tax,
        "budget": int8_budget,
    },
    "selected_for_service": selected_name,
    "adoption_decision": adoption_decision,
    "decision_scope": ship_decision,
    "fp32_rollback": "Re-export from recorded MODEL_SOURCE; weights stay outside GitHub.",
}

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
(reports_dir / "benchmark_results.json").write_text(
    json.dumps(json_safe(benchmark_results), ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(reports_dir / "service_smoke.json").write_text(
    json.dumps(
        json_safe({
            "result_label": RESULT_LABEL,
            "artefact_role": ARTEFACT_ROLE,
            "manifest": manifest.to_dict(),
            "tests": service_test_summary,
        }),
        ensure_ascii=False,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

benchmarks_draft = (
    f"# BENCHMARKS draft — {ARTEFACT_ROLE}\n\n"
    f"- Result label: `{RESULT_LABEL}`\n"
    f"- Decision scope: `{ship_decision}`\n"
    f"- Workload SHA-256: `{WORKLOAD_SHA256}`\n"
    f"- Device/provider: `{device}` / `CPUExecutionProvider`\n"
    f"- Warm-up/repetitions: {WARMUP}/{REPETITIONS}\n"
    f"- Memory method: {MEMORY_METHOD}\n"
    f"- PyTorch p95: {baseline_model_report['p95_ms']:.3f} ms\n"
    f"- ONNX FP32 p95: {fp32_ort_report['p95_ms']:.3f} ms\n"
    f"- ONNX FP32 quality tax: {fp32_quality_tax:.6f}\n"
    f"- INT8 available: {int8_status['available']}\n"
    f"- Selected for service: `{selected_name}`\n\n"
    f"- Adoption decision: `{adoption_decision}`\n\n"
    "> Replace this smoke draft with the complete BENCHMARKS template "
    "and full project workload before Gate D.\n"
)
(reports_dir / "BENCHMARKS_DRAFT.md").write_text(benchmarks_draft, encoding="utf-8")
print("REPORTS_WRITTEN", [
    "reports/benchmark_results.json",
    "reports/service_smoke.json",
    "reports/BENCHMARKS_DRAFT.md",
])


REPORTS_WRITTEN ['reports/benchmark_results.json', 'reports/service_smoke.json', 'reports/BENCHMARKS_DRAFT.md']


## بوابة Core | Core gate

Core يثبت: benchmark منضبط، ONNX parity، محاولة INT8 صادقة، API contract، canaries، وتقارير. **لا يثبت جودة مهمة** في Systems Smoke. عبور Gate D يحتاج Project Mode وworkload كاملة و`BENCHMARKS.md` النهائي.


In [17]:
core_checks = {
    "artefact_role_disclosed": ARTEFACT_ROLE in {"SYSTEMS_SMOKE", "PROJECT_ARTIFACT"},
    "budget_written_before_candidates": bool(BUDGET_PROVENANCE),
    "warmup_and_repetitions": WARMUP >= 1 and REPETITIONS >= 30,
    "bilingual_workload": {"ar", "en"}.issubset({row["language"] for row in rows}),
    "length_audited": set(length_report) >= {"p50_tokens", "p95_tokens", "max_tokens"},
    "baseline_tail_latency": set(baseline_model_report) >= {"p50_ms", "p95_ms", "p99_ms", "throughput_items_s"},
    "onnx_checked": FP32_ONNX.is_file(),
    "onnx_numerical_parity": fp32_parity["max_abs_logits_diff"] < 1e-3,
    "onnx_prediction_parity": fp32_parity["prediction_agreement"] == 1.0,
    "int8_attempt_honest": int8_status["attempted"] and (int8_status["available"] or bool(int8_status["error"])),
    "quality_tax_explicit": isinstance(fp32_quality_tax, float),
    "fastapi_contract": all(code in {200, 422} for code in service_test_summary.values() if isinstance(code, int)),
    "startup_canaries": len(service_test_summary["canaries"]) == 2,
    "reports_written": all((reports_dir / name).is_file() for name in [
        "benchmark_results.json", "service_smoke.json", "BENCHMARKS_DRAFT.md"
    ]),
    "large_artifacts_outside_repo": str(ARTIFACT_DIR).startswith("/content/"),
}
assert all(core_checks.values()), core_checks
print(core_checks)
print("DAY4_NOTEBOOK8_CORE=PASS")
if not PROJECT_MODE:
    print("NEXT_REQUIRED_FOR_GATE_D=RERUN_WITH_PROJECT_ARTIFACT_AND_FULL_WORKLOAD")


{'artefact_role_disclosed': True, 'budget_written_before_candidates': True, 'warmup_and_repetitions': True, 'bilingual_workload': True, 'length_audited': True, 'baseline_tail_latency': True, 'onnx_checked': True, 'onnx_numerical_parity': True, 'onnx_prediction_parity': True, 'int8_attempt_honest': True, 'quality_tax_explicit': True, 'fastapi_contract': True, 'startup_canaries': True, 'reports_written': True, 'large_artifacts_outside_repo': True}
DAY4_NOTEBOOK8_CORE=PASS
NEXT_REQUIRED_FOR_GATE_D=RERUN_WITH_PROJECT_ARTIFACT_AND_FULL_WORKLOAD


## بعد المختبر | After the lab

1. لبوابة Gate D أعد التشغيل بـ`PROJECT_MODE=True` وعلى workload المشروع كاملة، لا الدفعة الأولى فقط.
2. انقل **التقارير الصغيرة** إلى `reports/` في مستودعك؛ لا تنقل ONNX أو weights.
3. أكمل [`BENCHMARKS.md`](../templates/BENCHMARKS_TEMPLATE.md) وقرار Adopt/Reject/Rollback في `DECISIONS.md`.
4. أضف اختبارات benchmark/service إلى `tests/` وشغّلها.
5. استخدم commit: `perf: add optimized serving benchmark`.
6. ارجع إلى [Gate D وGate E](../day-04/05-lab-gates-submission.md).

**تنظيف اختياري بعد حفظ التقارير:** احذف `/content/bayan_day4_artifacts` فقط؛ لا تحذف Drive أو مجلد مشروعك. ستحتاج إعادة export إذا أردت تشغيل الخدمة بعد الحذف.


## 🔵 Explore و🟣 Distinction

ابدأ بعد Core وGate D فقط:

- **Explore:** قارِن `padding=max_length` وdynamic padding مع length buckets على workload نفسها.
- **Explore:** استخدم حزمة `optimum-onnx` الرسمية وقارن الناتج بالمسار المباشر.
- **Distinction:** قِس عدة batch sizes وارسم Pareto بين p95 وthroughput وquality tax.
- **Distinction:** أضف canary تكشف تغيّر label map أو preprocessing hash وتثبت أنها تفشل startup.

سجّل كل تجربة مستقلة ولا تنتقِ النتيجة الأفضل دون إظهار البدائل.


## Gate D — HTTP concurrency=16
قياس p99 عبر FastAPI TestClient على نموذج المشروع الفعلي.


In [ ]:
# Official Gate-D HTTP concurrency measurement: 16 simultaneous requests.
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter_ns

HTTP_CONCURRENCY = 16
HTTP_REPETITIONS = 30
http_payloads = [
    {"text": row["text"], "language": row["language"]}
    for row in rows
]

def one_http_request(index):
    payload = http_payloads[index % len(http_payloads)]
    start = perf_counter_ns()
    response = client.post("/v1/classify", json=payload)
    elapsed = (perf_counter_ns() - start) / 1_000_000
    assert response.status_code == 200, response.text
    return elapsed

# Warm-up is excluded.
with ThreadPoolExecutor(max_workers=HTTP_CONCURRENCY) as executor:
    list(executor.map(one_http_request, range(HTTP_CONCURRENCY)))

http_latencies = []
for repetition in range(HTTP_REPETITIONS):
    with ThreadPoolExecutor(max_workers=HTTP_CONCURRENCY) as executor:
        http_latencies.extend(
            executor.map(
                one_http_request,
                range(repetition * HTTP_CONCURRENCY, (repetition + 1) * HTTP_CONCURRENCY),
            )
        )

http_report = {
    "concurrency": HTTP_CONCURRENCY,
    "repetitions": HTTP_REPETITIONS,
    "requests": len(http_latencies),
    "p50_ms": float(np.percentile(http_latencies, 50)),
    "p95_ms": float(np.percentile(http_latencies, 95)),
    "p99_ms": float(np.percentile(http_latencies, 99)),
    "max_ms": float(np.max(http_latencies)),
}
benchmark_results["http_concurrency_16"] = http_report
(reports_dir / "benchmark_results.json").write_text(
    json.dumps(json_safe(benchmark_results), ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("HTTP_CONCURRENCY_16", http_report)
print("HTTP_P99_GATE", "PASS" if http_report["p99_ms"] <= 40.0 else "FAIL")
